# XDF Analyzer

Converts multi-stream XDF recordings into merged, analysis-ready CSVs. Two EmotiBit devices (Commander `_C`, Intelligence Support `_IS`) and a Neon eye tracker are exported from each XDF file, aligned by timestamp, labelled with video-reliability blocks, then joined with per-block survey scores.

## 1. Imports

In [28]:
import pyxdf
import pandas as pd
import numpy as np
from datetime import datetime
import os
import glob
import re


---

## 2. Core Functions

### Stream Export

Reads an XDF file and writes each stream to a separate CSV. Device source IDs determine the filename suffix: `_C` for Commander (`MD-V5-0001023`), `_I` for Intelligence Support (`MD-V6-0000405`).

In [29]:
def export_streams_to_csv(streams, output_dir="exported_streams"):

    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    exported_files = []
    for idx, stream in enumerate(streams):
        # Get stream metadata
        stream_name = stream['info']['name'][0]
        if stream_name == 'Keyboard':
            continue
        source_id = stream['info'].get('source_id', ['Unknown'])[0]
        if source_id is None:
            source_id = "Unknown"

        if source_id == "MD-V5-0001023":
            filename = f"{stream_name}_C.csv"
        elif source_id == "MD-V6-0000405":
            filename = f"{stream_name}_I.csv"
        else:
            filename = f"{stream_name}.csv"
        filepath = os.path.join(output_dir, filename)

        # Get data and timestamps
        data = stream['time_series']
        timestamps = stream['time_stamps']

        # Skip streams with no data
        if len(data) == 0:
            print(f"Skipping stream {idx} ({stream_name}): No data")
            continue

        # Convert data to numpy array if it's not already
        if not isinstance(data, np.ndarray):
            data = np.array(data)

        # Special handling for Neon Companion stream - export all channels with proper labels
        if "Neon" in stream_name and "Companion" in stream_name:
            # Build channel labels using label + eye suffix to disambiguate left/right duplicates
            channel_labels = []
            try:
                desc = stream['info'].get('desc')
                if desc and len(desc) > 0 and desc[0] is not None and isinstance(desc[0], dict):
                    channels_info = desc[0].get('channels')
                    if channels_info and len(channels_info) > 0 and channels_info[0] is not None:
                        channel_list = channels_info[0].get('channel', [])
                        for i, ch in enumerate(channel_list):
                            if isinstance(ch, dict) and 'label' in ch:
                                label = ch['label'][0] if isinstance(ch['label'], list) else ch['label']
                                eye = ch.get('eye')
                                if isinstance(eye, list):
                                    eye = eye[0] if eye else None
                                # Append eye suffix for per-eye channels to avoid duplicate column names
                                if eye and eye in ('left', 'right'):
                                    label = f"{label}_{eye}"
                                channel_labels.append(label)
                            else:
                                channel_labels.append(f'channel_{i}')
            except (AttributeError, KeyError, TypeError):
                pass

            # Pad with generic names if metadata doesn't cover all data channels
            while len(channel_labels) < data.shape[1]:
                channel_labels.append(f'channel_{len(channel_labels)}')

            # Create DataFrame with all channels
            df_dict = {'timestamp': timestamps}
            for i, label in enumerate(channel_labels):
                df_dict[label] = data[:, i]

            df = pd.DataFrame(df_dict)
            print(f"Exported Neon stream with {len(channel_labels)} channels: {', '.join(channel_labels)}")

        # Regular handling for other streams
        elif data.ndim == 1:
            # Single channel data
            df = pd.DataFrame({
                'timestamp': timestamps,
                'value': data
            })
        else:
            # Multi-channel data
            channel_count = data.shape[1] if len(data.shape) > 1 else 1
            df_dict = {'timestamp': timestamps}

            if channel_count == 1:
                # Treat as single channel even if it's 2D with 1 column
                df_dict['value'] = data.flatten()
            else:
                # Get channel labels if available (with safer handling)
                channel_labels = []
                try:
                    desc = stream['info'].get('desc')
                    if desc and len(desc) > 0 and desc[0] is not None and isinstance(desc[0], dict):
                        channels_info = desc[0].get('channels')
                        if channels_info and len(channels_info) > 0 and channels_info[0] is not None:
                            channel_list = channels_info[0].get('channel', [])
                            if channel_list:
                                for i in range(channel_count):
                                    if i < len(channel_list):
                                        label = channel_list[i].get('label', [f'channel_{i}'])
                                        if isinstance(label, list):
                                            label = label[0] if label else f'channel_{i}'
                                        channel_labels.append(label)
                                    else:
                                        channel_labels.append(f'channel_{i}')
                            else:
                                channel_labels = [f'channel_{i}' for i in range(channel_count)]
                        else:
                            channel_labels = [f'channel_{i}' for i in range(channel_count)]
                    else:
                        channel_labels = [f'channel_{i}' for i in range(channel_count)]
                except (AttributeError, KeyError, TypeError):
                    channel_labels = [f'channel_{i}' for i in range(channel_count)]

                # Add channel data to DataFrame
                for i in range(channel_count):
                    df_dict[channel_labels[i]] = data[:, i]

            df = pd.DataFrame(df_dict)

        # Export to CSV
        df.to_csv(filepath, index=False)
        exported_files.append(filepath)

        print(f"Exported stream {idx} ({stream_name}) to {filepath}")
        print(f"  Source ID: {source_id}")
        print(f"  Data shape: {data.shape}")
        print(f"  Sample count: {len(timestamps)}")
        if "Neon" in stream_name:
            print(f"  First timestamp: {timestamps[0]}")
        print()

    print(f"Successfully exported {len(exported_files)} streams to {output_dir}/")
    return exported_files

### Data Merging

`merge_csv_files` uses `PPG_GRN_C.csv` as the time base and nearest-timestamp joins (±21 ms) all other streams. Neon eye-tracking data is aggregated from 200 Hz → 25 Hz before merging. `_label_video_condition` is a standalone helper that applies the same video-segment labelling logic used inline by `merge_csv_files`.

In [30]:
def merge_csv_files(input_dir, output_path="merged_data.csv", use_ffill=True, tolerance=0.021, participant_id=None, condition=None, video_condition=None):
    """
    Merge CSV files using PPG_GRN_C as base, with nearest timestamp matching.

    Parameters:
    - input_dir: Directory containing CSV files
    - output_path: Path for the merged output file
    - use_ffill: Whether to forward fill missing values
    - tolerance: Maximum time difference for matching (seconds, default 0.021 = 21ms)
    - participant_id: ID to add as the first column in the output (optional)
    - condition: Condition to add as the second column in the output (optional)
    - video_condition: Video condition (1-6) for labeling Reliability sequences (optional)
    """
    import glob

    VIDEO_ORDERS = {
        1: ('LL', 'HH', 'M'),
        2: ('LL', 'M', 'HH'),
        3: ('HH', 'LL', 'M'),
        4: ('HH', 'M', 'LL'),
        5: ('M', 'LL', 'HH'),
        6: ('M', 'HH', 'LL')
    }

    csv_files = glob.glob(os.path.join(input_dir, "*.csv"))
    if not csv_files:
        print(f"No CSV files found in {input_dir}")
        return None
    print(f"Found {len(csv_files)} CSV files to merge")

    ppg_base = None
    other_files = []
    datasync_file = None

    for file_path in csv_files:
        filename = os.path.basename(file_path)
        if "PPG_GRN" in filename and "_C.csv" in filename:
            ppg_base = file_path
            print(f"Found PPG_GRN base file: {filename}")
        elif "DataSyncMarker" in filename:
            datasync_file = file_path
        elif "Keyboard" in filename:
            pass
        else:
            other_files.append(file_path)

    if ppg_base is None:
        print("No PPG_GRN_C.csv file found to use as base!")
        return None

    try:
        base_df = pd.read_csv(ppg_base)
        base_filename = os.path.basename(ppg_base)
        print(f"Base file: {base_filename} ({len(base_df)} rows)")
        base_df['timestamp'] = base_df['timestamp'].round(3)

        merged_df = pd.DataFrame()
        if participant_id is not None:
            merged_df['ID'] = [participant_id] * len(base_df)
        if condition is not None:
            merged_df['Condition'] = [condition] * len(base_df)
        merged_df['timestamp'] = base_df['timestamp']

        if datasync_file is not None:
            try:
                datasync_df = pd.read_csv(datasync_file)
                datasync_filename = os.path.basename(datasync_file)

                if 'timestamp' in datasync_df.columns:
                    print(f"\nProcessing {datasync_filename} (adding after timestamp)...")
                    datasync_df['timestamp'] = datasync_df['timestamp'].round(3)
                    print(f"  File timestamps: {datasync_df['timestamp'].min():.3f} to {datasync_df['timestamp'].max():.3f}")
                    data_columns = [col for col in datasync_df.columns if col != 'timestamp']
                    datasync_df = datasync_df.sort_values('timestamp').reset_index(drop=True)

                    if 'channel_0' in data_columns:
                        print(f"  Expanding DataSyncMarker events into time ranges...")
                        print(f"  Original DataSyncMarker events: {len(datasync_df)} events")
                        marker_events = datasync_df[['timestamp', 'channel_0']].sort_values('timestamp').copy()
                        print(f"  DataSyncMarker events:")
                        for idx, row in marker_events.iterrows():
                            print(f"    t={row['timestamp']:.3f}, value={row['channel_0']}")

                        merged_df = pd.merge_asof(merged_df,
                                                  marker_events[['timestamp', 'channel_0']],
                                                  on='timestamp',
                                                  direction='backward')

                        # Compute Reliability labels from raw channel_0 (1=video, 2=no video)
                        if video_condition is not None and video_condition in VIDEO_ORDERS:
                            video_order = VIDEO_ORDERS[video_condition]
                            reliability_labels = []
                            video_segment_num = 0
                            prev_state = None
                            for state in merged_df['channel_0']:
                                state_int = int(state) if pd.notna(state) else None
                                if state_int != prev_state:
                                    if state_int == 1:
                                        video_segment_num += 1
                                    prev_state = state_int
                                if state_int == 1 and 1 <= video_segment_num <= len(video_order):
                                    reliability_labels.append(video_order[video_segment_num - 1])
                                elif state_int == 2:
                                    reliability_labels.append("No Video")
                                else:
                                    reliability_labels.append(None)
                            merged_df['Reliability'] = reliability_labels
                            print(f"  Added Reliability column using video condition {video_condition}: {video_order}")

                        # Keep DataSyncMarker as original binary values (1=video, 2=no video)
                        merged_df = merged_df.rename(columns={'channel_0': 'DataSyncMarker'})

                        print(f"  Added DataSyncMarker column after timestamp")
                        matched_count = merged_df['DataSyncMarker'].notna().sum()
                        print(f"    DataSyncMarker: {matched_count}/{len(merged_df)} matches ({100*matched_count/len(merged_df):.1f}%)")

                        value_counts = merged_df['DataSyncMarker'].value_counts()
                        print(f"  DataSyncMarker value distribution:")
                        for value, count in value_counts.items():
                            print(f"    {value}: {count} rows ({100*count/len(merged_df):.1f}%)")

                    remaining_columns = [col for col in data_columns if col != 'channel_0']
                    if remaining_columns:
                        remaining_df = datasync_df[['timestamp'] + remaining_columns].copy()
                        file_prefix = datasync_filename.replace('.csv', '').replace('stream_', '')
                        for col in remaining_columns:
                            remaining_df = remaining_df.rename(columns={col: f"{file_prefix}_{col}"})
                        temp_path = "temp_datasync_remaining.csv"
                        remaining_df.to_csv(temp_path, index=False)
                        other_files.append(temp_path)

            except Exception as e:
                print(f"Error processing DataSyncMarker file {datasync_file}: {str(e)}")

        base_prefix = base_filename.replace('.csv', '').replace('stream_', '')
        data_columns = [col for col in base_df.columns if col != 'timestamp']
        for col in data_columns:
            merged_df[f"{base_prefix}_{col}"] = base_df[col]
        merged_df = merged_df.sort_values('timestamp').reset_index(drop=True)

    except Exception as e:
        print(f"Error reading base file {ppg_base}: {str(e)}")
        return None

    print(f"Base PPG timestamps: {merged_df['timestamp'].min():.3f} to {merged_df['timestamp'].max():.3f}")

    for i, file_path in enumerate(other_files):
        try:
            filename = os.path.basename(file_path)
            if filename == "temp_datasync_remaining.csv":
                df = pd.read_csv(file_path)
                if os.path.exists(file_path):
                    os.remove(file_path)
            else:
                df = pd.read_csv(file_path)

            if 'timestamp' not in df.columns:
                print(f"Skipping {filename}: No 'timestamp' column found")
                continue

            df['timestamp'] = df['timestamp'].round(3)
            print(f"\nProcessing {filename}...")
            print(f"  File timestamps: {df['timestamp'].min():.3f} to {df['timestamp'].max():.3f}")

            if "Neon" in filename:
                neon_columns = [col for col in df.columns if col != 'timestamp']
                if not neon_columns:
                    print(f"Skipping {filename}: No data columns found")
                    continue
                print(f"  Downsampling {len(neon_columns)} Neon eye-tracking channels from 200Hz to 25Hz using aggregation...")
                min_time = df['timestamp'].min()
                max_time = df['timestamp'].max()
                bin_size = 0.04
                time_bins = np.round(np.arange(min_time, max_time + bin_size, bin_size), 3)
                df['time_bin'] = pd.cut(df['timestamp'], bins=time_bins, labels=time_bins[:-1], include_lowest=True)
                aggregated_data = []
                for bin_center in time_bins[:-1]:
                    bin_data = df[df['time_bin'] == bin_center]
                    if len(bin_data) > 0:
                        row_data = {'timestamp': bin_center}
                        for col in neon_columns:
                            valid_values = bin_data[col].dropna()
                            row_data[col] = valid_values.mean() if len(valid_values) > 0 else np.nan
                        aggregated_data.append(row_data)
                df = pd.DataFrame(aggregated_data)
                print(f"  After aggregation: {len(df)} samples ({len(neon_columns)} channels)")
                for col in neon_columns:
                    valid_count = df[col].notna().sum()
                    print(f"    {col}: {valid_count}/{len(df)} valid aggregated values ({100*valid_count/len(df):.1f}%)")

            data_columns = [col for col in df.columns if col != 'timestamp' and col != 'time_bin']
            df = df.sort_values('timestamp').reset_index(drop=True)
            df_for_merge = df.copy()

            if filename != "temp_datasync_remaining.csv":
                file_prefix = filename.replace('.csv', '').replace('stream_', '')
                for col in data_columns:
                    df_for_merge = df_for_merge.rename(columns={col: f"{file_prefix}_{col}"})

            # Track which columns are new after this merge
            cols_before = set(merged_df.columns)

            if filename == "temp_datasync_remaining.csv":
                print(f"  Expanding DataSyncMarker_channel_1 events into time ranges...")
                merged_df = pd.merge_asof(merged_df, df_for_merge, on='timestamp', direction='backward')
            else:
                merged_df = pd.merge_asof(merged_df, df_for_merge, on='timestamp', direction='nearest', tolerance=tolerance)

            new_cols = [col for col in merged_df.columns if col not in cols_before]
            print(f"  Added {len(new_cols)} column(s) from {filename}")
            for col in new_cols:
                matched_count = merged_df[col].notna().sum()
                print(f"    {col}: {matched_count}/{len(merged_df)} matches ({100*matched_count/len(merged_df):.1f}%)")

        except Exception as e:
            print(f"Error processing {filename}: {str(e)}")
            continue

    # Convert DataSyncMarker_channel_1 from UNIX time to Date and Time columns
    if 'DataSyncMarker_channel_1' in merged_df.columns:
        print(f"\nCalculating Date and Time columns using DataSyncMarker and EmotiBit timestamps...")
        valid_sync_timestamps = merged_df['DataSyncMarker_channel_1'].notna()

        if valid_sync_timestamps.any():
            try:
                first_sync_idx = merged_df[valid_sync_timestamps].index[0]
                first_sync_unix = merged_df.loc[first_sync_idx, 'DataSyncMarker_channel_1']
                sync_emotibit_time = merged_df.loc[first_sync_idx, 'timestamp']

                if 1577836800 <= first_sync_unix <= 1893456000:
                    import pytz
                    utc_datetime = pd.to_datetime(first_sync_unix, unit='s', utc=True)
                    est_tz = pytz.timezone('US/Eastern')
                    initial_datetime_est = utc_datetime.tz_convert(est_tz)

                    print(f"Initial sync point at row {first_sync_idx}:")
                    print(f"  UNIX timestamp: {first_sync_unix}")
                    print(f"  EST time: {initial_datetime_est}")
                    print(f"  Corresponding EmotiBit timestamp: {sync_emotibit_time}")

                    time_diffs = merged_df['timestamp'] - sync_emotibit_time
                    calculated_datetimes = initial_datetime_est.tz_localize(None) + pd.to_timedelta(time_diffs, unit='s')

                    date_col = calculated_datetimes.dt.strftime('%Y-%m-%d')
                    time_col = calculated_datetimes.dt.strftime('%H:%M:%S.%f').str[:-3]

                    columns = list(merged_df.columns)
                    if 'Condition' in columns:
                        insert_pos = columns.index('Condition') + 1
                    elif 'ID' in columns:
                        insert_pos = columns.index('ID') + 1
                    else:
                        insert_pos = 0

                    columns.insert(insert_pos, 'Date')
                    columns.insert(insert_pos + 1, 'Time')
                    merged_df['Date'] = date_col
                    merged_df['Time'] = time_col
                    merged_df = merged_df[columns]

                    print(f"Successfully created Date and Time columns for all {len(merged_df)} rows (EST timezone)")
                    print(f"Duration: {(calculated_datetimes.max() - calculated_datetimes.min()).total_seconds():.2f} seconds")

                    start_idx = max(0, first_sync_idx - 2)
                    end_idx = min(len(merged_df), first_sync_idx + 3)
                    print(f"\nExample rows around sync point (row {first_sync_idx}):")
                    for i in range(start_idx, end_idx):
                        marker = " <- SYNC POINT" if i == first_sync_idx else ""
                        time_diff = merged_df.loc[i, 'timestamp'] - sync_emotibit_time
                        print(f"  Row {i}: EmotiBit={merged_df.loc[i, 'timestamp']:.3f}, diff={time_diff:+.3f}s, Time={time_col.iloc[i]}{marker}")
                else:
                    print(f"First DataSyncMarker value {first_sync_unix} is outside expected range (2020-2030)")

            except Exception as e:
                print(f"Error calculating Date and Time columns: {str(e)}")
        else:
            print("No valid DataSyncMarker_channel_1 values found")

    # Apply forward fill
    if use_ffill:
        print(f"\nApplying forward fill to missing values...")
        missing_before = merged_df.isnull().sum().sum()

        exclude_cols = ['timestamp', 'ID', 'Condition', 'Date', 'Time']
        data_columns = [col for col in merged_df.columns if col not in exclude_cols]
        merged_df[data_columns] = merged_df[data_columns].ffill()

        missing_after = merged_df.isnull().sum().sum()
        print(f"Missing values before ffill: {missing_before}")
        print(f"Missing values after ffill: {missing_after}")
        print(f"Filled {missing_before - missing_after} missing values")

    # Rename columns:
    #   - strip _value suffix
    #   - TEMP1_X -> TEMP_X
    #   - _I suffix -> _IS
    rename_map = {}
    for col in merged_df.columns:
        new_col = col
        if new_col.endswith('_value'):
            new_col = new_col[:-6]
        new_col = new_col.replace('TEMP1_', 'TEMP_')
        if new_col.endswith('_I'):
            new_col = new_col + 'S'
        if new_col != col:
            rename_map[col] = new_col
    merged_df = merged_df.rename(columns=rename_map)
    print(f"\nRenamed {len(rename_map)} columns: {rename_map}")

    # Reorder columns — Neon columns are detected dynamically and placed after EmotiBit columns
    neon_cols = [col for col in merged_df.columns if 'Neon' in col]
    desired_order = [
        'ID', 'Condition', 'Date', 'Time', 'timestamp',
        'DataSyncMarker', 'DataSyncMarker_channel_1', 'Reliability',
        'PPG_GRN_C', 'PPG_RED_C', 'PPG_IR_C', 'EDA_C', 'HR_C', 'TEMP_C', 'THERM_C',
    ] + neon_cols + [
        'PPG_GRN_IS', 'PPG_RED_IS', 'PPG_IR_IS', 'EDA_IS', 'HR_IS', 'TEMP_IS', 'THERM_IS'
    ]
    ordered_cols = [col for col in desired_order if col in merged_df.columns]
    remaining_cols = [col for col in merged_df.columns if col not in desired_order]
    if remaining_cols:
        print(f"Note: extra columns appended at end: {remaining_cols}")
    merged_df = merged_df[ordered_cols + remaining_cols]

    merged_df.to_csv(output_path, index=False)

    print(f"\n=== MERGE SUMMARY ===")
    print(f"Final dataset: {len(merged_df)} rows, {len(merged_df.columns)} columns")
    print(f"Columns: {list(merged_df.columns)}")
    print(f"Time range: {merged_df['timestamp'].min():.3f} to {merged_df['timestamp'].max():.3f}")
    print(f"Duration: {merged_df['timestamp'].max() - merged_df['timestamp'].min():.2f} seconds")
    if participant_id is not None:
        print(f"Participant ID: {participant_id}")
    if condition is not None:
        print(f"Condition: {condition}")
    if video_condition is not None:
        print(f"Video condition: {video_condition}")

    print(f"\n=== FINAL DATA COMPLETENESS ===")
    for col in merged_df.columns:
        if col not in ['timestamp', 'ID', 'Condition', 'Date', 'Time']:
            non_null_count = merged_df[col].notna().sum()
            completeness = (non_null_count / len(merged_df)) * 100
            print(f"{col}: {non_null_count}/{len(merged_df)} ({completeness:.1f}% complete)")

    return merged_df

In [31]:
def _label_video_condition(binary_sequence, video_condition):
    """
    Label binary sequence based on video condition and playback order.
    
    Parameters:
    - binary_sequence: Series of 1s and 2s (1=video playback, 2=no video)
    - video_condition: Integer 1-6 indicating which of the 6 video orders from ExperimentPlayer
    
    Returns:
    - Series with labels: "HH", "LL", "M", "No Video", or original binary values
    """

    video_orders = {
        1: ('LL', 'HH', 'M'),
        2: ('LL', 'M', 'HH'),
        3: ('HH', 'LL', 'M'),
        4: ('HH', 'M', 'LL'),
        5: ('M', 'LL', 'HH'),
        6: ('M', 'HH', 'LL')
    }

    if video_condition not in video_orders:
        return binary_sequence

    order = video_orders[video_condition]
    seq_array = binary_sequence.values
    labels = seq_array.astype(object)

    # Identify which video block each index belongs to by counting transitions into state 1
    video_segment_num = 0
    prev_state = None

    for i in range(len(seq_array)):
        val = seq_array[i]
        val_int = int(val) if pd.notna(val) else None

        # Detect transition into a new video block
        if val_int != prev_state:
            if val_int == 1:
                video_segment_num += 1
            prev_state = val_int

        if val_int == 2:
            labels[i] = "No Video"
        elif val_int == 1:
            if 1 <= video_segment_num <= len(order):
                labels[i] = order[video_segment_num - 1]
            else:
                labels[i] = val  # more segments than expected, leave as-is
        else:
            # Forward fill for NaN or unexpected values
            if i > 0:
                labels[i] = labels[i - 1]

    return pd.Series(labels, index=binary_sequence.index)

### Restart Fix (ID25, ID46)

Two participants' sessions were restarted partway through a block: the experiment
player always restarts from block 1, so the `DataSyncMarker` stream gets extra `==1`
runs (junk replays of earlier blocks) plus one `==3` "skip" marker per replayed block,
sandwiched between the aborted attempt at the restarted block and its real continuation.

`label_restart_participant` keeps every row — unlike the row-dropping approach used
earlier, the restart window is labeled `Reliability == "Restart"` in place. Block
numbering for the surviving runs is derived by counting only the *kept* `==1` runs;
junk runs inside the window never advance the counter, so the real block lands on the
correct `video_order` entry regardless of how many junk attempts preceded it.

`RESTART_FIXES` maps `participant_id -> restarted block number K` (1-indexed) and is
the single source of truth for which participants get this treatment.

In [32]:
VIDEO_ORDERS = {
    1: ('LL', 'HH', 'M'),
    2: ('LL', 'M', 'HH'),
    3: ('HH', 'LL', 'M'),
    4: ('HH', 'M', 'LL'),
    5: ('M', 'LL', 'HH'),
    6: ('M', 'HH', 'LL')
}

RESTART_FIXES = {25: 2, 46: 3}   # participant_id -> restarted block number K


def label_restart_participant(df, restart_block, video_condition):
    """
    Relabel Reliability for a participant whose session was restarted partway
    through block `restart_block` (1-indexed). Keeps every row: the aborted
    attempt, replay(s), and skip marker(s) are tagged "Restart"; block numbers
    for the surviving runs come from counting only the KEPT DataSyncMarker==1
    runs (junk runs inside the restart window never advance the counter).
    """
    df = df.reset_index(drop=True)

    if video_condition not in VIDEO_ORDERS:
        print(f"WARNING: unknown video_condition {video_condition}; cannot determine video order")
        return df
    order = VIDEO_ORDERS[video_condition]

    def _state(v):
        if pd.isna(v):
            return None
        if v != 0 and abs(v) < 1e-6:
            return 'err'
        return int(round(v))

    states = df['DataSyncMarker'].map(_state)

    # Contiguous runs as (state, start_pos, end_pos)
    runs = []
    run_state, run_start = states.iloc[0], 0
    for i in range(1, len(states)):
        s = states.iloc[i]
        if s != run_state:
            runs.append((run_state, run_start, i - 1))
            run_state, run_start = s, i
    runs.append((run_state, run_start, len(states) - 1))

    one_runs = [r for r in runs if r[0] == 1]
    three_runs = [r for r in runs if r[0] == 3]

    print(f"DataSyncMarker==1 runs: {len(one_runs)}")
    for idx, (s, start, end) in enumerate(one_runs, 1):
        print(f"  run {idx}: rows {start}-{end} (n={end - start + 1})")
    print(f"DataSyncMarker==3 runs: {len(three_runs)}")
    for idx, (s, start, end) in enumerate(three_runs, 1):
        print(f"  run {idx}: rows {start}-{end} (n={end - start + 1})")

    if len(one_runs) < restart_block:
        print(f"WARNING: expected at least {restart_block} DataSyncMarker==1 run(s), found {len(one_runs)} -- returning df unchanged")
        return df
    if not three_runs:
        print("WARNING: no DataSyncMarker==3 run found -- cannot determine restart window; returning df unchanged")
        return df

    delete_start = one_runs[restart_block - 1][1]
    delete_end = three_runs[-1][2]
    if delete_end <= delete_start:
        print(f"WARNING: delete_end ({delete_end}) <= delete_start ({delete_start}); restart window invalid -- returning df unchanged")
        return df
    print(f"Restart window: rows {delete_start}-{delete_end} (n={delete_end - delete_start + 1})")

    window = set(range(delete_start, delete_end + 1))
    labels = [None] * len(df)
    block_segment = 0
    prev_kept_state = None
    none_outside_window = 0
    other_outside_window = 0

    for i, s in enumerate(states):
        if i in window:
            labels[i] = "Restart"
            continue
        if s == 1:
            if prev_kept_state != 1:
                block_segment += 1
            labels[i] = order[block_segment - 1] if 1 <= block_segment <= len(order) else None
        elif s == 2:
            labels[i] = "No Video"
        else:
            labels[i] = None
            if s is None:
                none_outside_window += 1
            else:
                other_outside_window += 1
        prev_kept_state = s

    if block_segment != len(order):
        print(f"WARNING: counted {block_segment} kept DataSyncMarker==1 run(s) outside the window, expected {len(order)}")
    if none_outside_window:
        print(f"  ({none_outside_window} row(s) outside the window had no marker yet -- expected before the first block starts)")
    if other_outside_window:
        print(f"WARNING: {other_outside_window} row(s) outside the restart window had an unexpected non-1/2 DataSyncMarker state")

    df['Reliability'] = labels

    n_restart = sum(1 for l in labels if l == "Restart")
    print(f"Tagged {n_restart} row(s) as 'Restart'")

    return df

---

## 3. Configuration

Paths for input XDF files, metadata, survey scores, and output CSVs. Update these before running the pipeline below.

In [33]:
# --- Batch Configuration ---
xdf_folder        = "/Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data"   # directory containing [number]_C.xdf files
metadata_csv      = "/Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Metadata.csv" # CSV with columns: Team_ID, condition, video_condition
survey_csv        = "/Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/mdmt_physio.csv"   # CSV with columns: Team_ID, Role, Condition, Block, ...
physio_output_path = "all_participants_physio.csv"   # raw physiological data (no survey columns)
output_path       = "all_participants_combined.csv"  # final output with survey columns merged in


---

## 4. Pipeline Functions

### Batch Processing

Discovers all `[ID]_C.xdf` files, reads per-participant metadata, and orchestrates the full export → merge pipeline.

In [34]:
def process_batch(xdf_folder, metadata_csv, output_path="all_participants_combined.csv"):
    """
    Process all [number]_C.xdf files in xdf_folder in ascending ID order.

    For each file the matching row in metadata_csv (keyed on Team_ID) supplies
    condition and video_condition.  use_ffill=True and tolerance=0.021 are
    fixed and not read from the metadata.

    Intermediate outputs written to xdf_folder:
      <ID>_Data/             — per-participant stream CSVs
      merged_data_<ID>.csv   — per-participant merged CSV

    Parameters
    ----------
    xdf_folder   : str  — directory containing [number]_C.xdf files
    metadata_csv : str  — CSV with at least columns Team_ID, condition, video_condition
    output_path  : str  — path for the combined output CSV
    """
    meta = pd.read_csv(metadata_csv)
    meta['Team_ID'] = meta['Team_ID'].astype(int)
    meta = meta.set_index('Team_ID')
    dupes = meta.index[meta.index.duplicated()].unique().tolist()
    if dupes:
        print(f'WARNING: duplicate Team_IDs in metadata (keeping first): {dupes}')
        meta = meta[~meta.index.duplicated(keep='first')]

    # Discover [number]_C.xdf files and sort by numeric ID
    xdf_files = []
    for fname in os.listdir(xdf_folder):
        m = re.match(r'^(\d+)_C\.xdf$', fname)
        if m:
            xdf_files.append((int(m.group(1)), os.path.join(xdf_folder, fname)))
    xdf_files.sort()

    if not xdf_files:
        print(f"No [number]_C.xdf files found in {xdf_folder}")
        return None

    print(f"Found {len(xdf_files)} file(s): IDs {[pid for pid, _ in xdf_files]}")

    all_dfs = []

    for participant_id, xdf_path in xdf_files:
        if participant_id not in meta.index:
            print(f"\nWARNING: No metadata row for ID {participant_id} — skipping")
            continue

        row             = meta.loc[participant_id]
        condition       = row['condition']
        video_condition = int(row['video_condition'])

        print(f"\n{'='*60}")
        print(f"Processing ID {participant_id}  ({os.path.basename(xdf_path)})")
        print(f"  condition={condition}  video_condition={video_condition}")
        print('='*60)

        temp_dir     = os.path.join(xdf_folder, f"{participant_id}_Data")
        per_part_csv = os.path.join(xdf_folder, f"merged_data_{participant_id}.csv")

        try:
            streams, _ = pyxdf.load_xdf(xdf_path)
            export_streams_to_csv(streams, output_dir=temp_dir)
            df = merge_csv_files(
                input_dir       = temp_dir,
                output_path     = per_part_csv,
                use_ffill       = True,
                tolerance       = 0.021,
                participant_id  = participant_id,
                condition       = condition,
                video_condition = video_condition,
            )
            if df is not None:
                if participant_id in RESTART_FIXES:
                    restart_block = RESTART_FIXES[participant_id]
                    print(f"\nApplying restart fix for ID {participant_id} (restarted at block {restart_block})...")
                    df = label_restart_participant(df, restart_block, video_condition)
                    df.to_csv(per_part_csv, index=False)
                    print(f"Re-saved relabeled CSV: {per_part_csv}")
                all_dfs.append(df)
        except Exception as e:
            print(f"ERROR: participant {participant_id}: {e}")
            continue

    if not all_dfs:
        print("No data to combine.")
        return None

    combined = pd.concat(all_dfs, ignore_index=True)
    combined.to_csv(output_path, index=False)

    processed_ids = [int(df['ID'].iloc[0]) for df in all_dfs]
    print(f"\n{'='*60}")
    print(f"BATCH COMPLETE — {len(all_dfs)} participant(s): {processed_ids}")
    print(f"Total rows : {len(combined)}")
    print(f"Output     : {output_path}")
    print('='*60)

    return combined

### Reprocess a Single Participant

For fixing one participant (e.g. ID25) without re-running the whole batch. Reuses the
already-exported `<ID>_Data/` stream CSVs unless `force_reexport=True`. Patches the
result into `physio_output_path` by replacing that participant's old rows — you still
need to re-run Step 2 (`merge_survey_data`) afterward since block/survey data depends
on the corrected rows.

In [35]:
def reprocess_participant(participant_id, xdf_folder, metadata_csv, physio_output_path, force_reexport=False):
    """
    Re-run export+merge for a single participant and patch the result into
    the existing physio_output_path CSV (replacing that participant's old rows).
    """
    meta = pd.read_csv(metadata_csv)
    meta['Team_ID'] = meta['Team_ID'].astype(int)
    meta = meta.set_index('Team_ID')
    row             = meta.loc[participant_id]
    condition       = row['condition']
    video_condition = int(row['video_condition'])

    xdf_path     = os.path.join(xdf_folder, f"{participant_id}_C.xdf")
    temp_dir     = os.path.join(xdf_folder, f"{participant_id}_Data")
    per_part_csv = os.path.join(xdf_folder, f"merged_data_{participant_id}.csv")

    if force_reexport or not os.path.isdir(temp_dir):
        print(f"Exporting streams from {xdf_path}...")
        streams, _ = pyxdf.load_xdf(xdf_path)
        export_streams_to_csv(streams, output_dir=temp_dir)
    else:
        print(f"Using existing exported streams in {temp_dir} (force_reexport=False)")

    df = merge_csv_files(
        input_dir       = temp_dir,
        output_path     = per_part_csv,
        use_ffill       = True,
        tolerance       = 0.021,
        participant_id  = participant_id,
        condition       = condition,
        video_condition = video_condition,
    )
    if df is None:
        print(f"ERROR: merge_csv_files returned None for participant {participant_id}")
        return None

    if participant_id in RESTART_FIXES:
        restart_block = RESTART_FIXES[participant_id]
        print(f"\nApplying restart fix for ID {participant_id} (restarted at block {restart_block})...")
        df = label_restart_participant(df, restart_block, video_condition)
        df.to_csv(per_part_csv, index=False)
        print(f"Re-saved relabeled CSV: {per_part_csv}")

    physio = pd.read_csv(physio_output_path, low_memory=False)
    before = len(physio)
    physio = physio[physio['ID'] != participant_id]
    combined = pd.concat([physio, df], ignore_index=True)
    combined.to_csv(physio_output_path, index=False)
    print(f"\nPatched ID {participant_id} into {physio_output_path}: "
          f"{before} -> {len(combined)} total rows ({len(df)} rows for this participant)")

    return combined

### Survey Integration

Left-merges per-block survey scores into the physiological time-series. Block boundaries are detected from `DataSyncMarker` transitions; Commander scores get suffix `_C`, Intelligence Support scores get `_IS`.

In [50]:
def merge_survey_data(combined_df, survey_df):
    """
    Merge per-block survey scores into the physiological time-series DataFrame.

    Block derivation (per ID × Condition group):
      Scan DataSyncMarker in row order; each transition into 1.0 starts the
      next block. Block counter starts at 1; rows before the first 1.0 marker
      belong to Block 1.  Rows where Reliability == "Restart" (see
      label_restart_participant) are skipped entirely -- they neither count as
      a transition nor advance the "last seen state," and are tagged with
      Block = -1 so they never match a real survey block. This keeps block
      numbering correct for the surviving rows of a restarted participant
      (e.g. ID25, ID46) without needing any participant-specific branching
      here.  The temporary Block column is dropped from the output.

    Survey roles and column suffixes:
      "Commander"            → non-key columns suffixed _C
      "Intelligence Support" → non-key columns suffixed _IS

    Key columns (not suffixed): Team_ID, Role, Condition, Block, Participant_ID
    Join: combined_df[ID, Condition, Block] ↔ survey[Team_ID, Condition, Block]

    Video_Condition is coalesced from _C / _IS into a single column placed
    between DataSyncMarker_channel_1 and Reliability.

    The function is idempotent: any survey columns already present in
    combined_df (from a prior run) are dropped before merging, including
    _x/_y duplicates from previous bad runs.
    """
    KEY_COLS = {'Team_ID', 'Role', 'Condition', 'Block', 'Participant_ID'}
    ROLE_MAP = {'Commander': '_C', 'Intelligence Support': '_IS'}
    survey_val_cols = [c for c in survey_df.columns if c not in KEY_COLS]

    # Drop ALL variants of pre-existing survey columns (exact, _x, _y) so
    # re-runs on a previously-merged CSV don't cause MergeError duplicates.
    stale_present = [
        col for col in combined_df.columns
        if any(
            col in (f"{c}{sfx}", f"{c}{sfx}_x", f"{c}{sfx}_y")
            for c in survey_val_cols for sfx in ('_C', '_IS')
        ) or col == 'Video_Condition'
    ]
    if stale_present:
        print(f"  Dropping {len(stale_present)} pre-existing survey column(s): {stale_present}")
        combined_df = combined_df.drop(columns=stale_present)

    # --- Derive Block per (ID, Condition) group so the counter resets for each participant ---
    def _derive_blocks(grp):
        nums, b, prev, first = [], 1, None, False
        has_reliability = 'Reliability' in grp.columns
        for i in range(len(grp)):
            m = grp['DataSyncMarker'].iloc[i]
            if has_reliability and grp['Reliability'].iloc[i] == 'Restart':
                nums.append(-1)
                continue
            if pd.notna(m) and float(m) == 1.0 and prev != 1.0:
                if first:
                    b += 1
                else:
                    first = True
            if pd.notna(m):
                prev = float(m)
            nums.append(b)
        return pd.Series(nums, index=grp.index, dtype=int)

    result = combined_df.copy()
    result['Block'] = (
        result.groupby(['ID', 'Condition'], sort=False, group_keys=False)
              .apply(_derive_blocks, include_groups=False)
    )

    # --- Build and left-merge one suffixed table per role ---
    for role, suffix in ROLE_MAP.items():
        role_rows = survey_df[survey_df['Role'] == role].copy()
        if role_rows.empty:
            print(f"  WARNING: no survey rows for role '{role}'")
            continue

        role_rows = role_rows.rename(columns={c: f"{c}{suffix}" for c in survey_val_cols})
        role_rows = role_rows.rename(columns={'Team_ID': 'ID'})
        role_rows = role_rows.drop(columns=[c for c in ('Role', 'Participant_ID') if c in role_rows.columns])

        suffixed = [f"{c}{suffix}" for c in survey_val_cols if f"{c}{suffix}" in role_rows.columns]
        role_rows = role_rows[['ID', 'Condition', 'Block'] + suffixed]

        result = result.merge(role_rows, on=['ID', 'Condition', 'Block'], how='left')

    result = result.drop(columns=['Block'])

    # Coalesce Video_Condition_C / Video_Condition_IS into one column placed
    # between DataSyncMarker_channel_1 and Reliability.
    vc_cols = [c for c in ('Video_Condition_C', 'Video_Condition_IS') if c in result.columns]
    if vc_cols:
        result['Video_Condition'] = result[vc_cols[0]]
        for c in vc_cols[1:]:
            result['Video_Condition'] = result['Video_Condition'].combine_first(result[c])
        result = result.drop(columns=vc_cols)

        cols = list(result.columns)
        cols.remove('Video_Condition')
        if 'DataSyncMarker_channel_1' in cols:
            insert_pos = cols.index('DataSyncMarker_channel_1') + 1
        elif 'Reliability' in cols:
            insert_pos = cols.index('Reliability')
        else:
            insert_pos = len(cols)
        cols.insert(insert_pos, 'Video_Condition')
        result = result[cols]

    # --- Summary ---
    tmp = combined_df.copy()
    tmp['Block'] = (
        tmp.groupby(['ID', 'Condition'], sort=False, group_keys=False)
           .apply(_derive_blocks, include_groups=False)
    )
    print("\n=== SURVEY MERGE SUMMARY ===")
    for (uid, cond), grp in tmp.groupby(['ID', 'Condition']):
        n_blocks = grp.loc[grp['Block'] != -1, 'Block'].nunique()
        n_restart = (grp['Block'] == -1).sum()
        extra = f"  ({n_restart:,} row(s) tagged Restart, excluded)" if n_restart else ""
        print(f"  ID {int(uid)} ({cond}): {n_blocks} block(s) detected{extra}")
    for role, suffix in ROLE_MAP.items():
        sample = next((c for c in result.columns if c.endswith(suffix)), None)
        if sample:
            n = result[sample].notna().sum()
            print(f"  Rows with {role} data: {n:,} / {len(result):,}")

    return result

---

## 5. Run Pipeline

### Step 1 — Process XDF Files

Loads each XDF file, exports all streams to per-participant CSVs, and merges them into a single physio file (`physio_output_path`).

In [37]:
combined_df = process_batch(xdf_folder, metadata_csv, physio_output_path)


Found 52 file(s): IDs [4, 6, 7, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 60]

Processing ID 4  (4_C.xdf)
  condition=Distrust Priming  video_condition=1


Stream 5: No samples in clock offsets (0, 0), skipping...
Stream 17: No samples in clock offsets (0, 0), skipping...
Stream 17: Calculated effective sampling rate 158.9756 Hz is different from specified rate 200.0000 Hz.


Exported stream 0 (TEMP1) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data/4_Data/TEMP1_I.csv
  Source ID: MD-V6-0000405
  Data shape: (21299, 1)
  Sample count: 21299

Exported stream 2 (HR) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data/4_Data/HR_C.csv
  Source ID: MD-V5-0001023
  Data shape: (2316, 1)
  Sample count: 2316

Exported stream 3 (DataSyncMarker) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data/4_Data/DataSyncMarker.csv
  Source ID: 12345
  Data shape: (7, 2)
  Sample count: 7

Exported stream 4 (EDA) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data/4_Data/EDA_C.csv
  Source ID: MD-V5-0001023
  Data shape: (42591, 1)
  Sample count: 42591

Exported stream 5 (HR) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data/4_Data/HR_I.csv
  Source ID: MD-V6-0000405
  Data shape: (1815, 1)
  Sample count: 1815

Exported stream 6 (PPG_RED) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data/4_Data/PPG_RED_C.csv

Stream 12: Calculated effective sampling rate 142.4540 Hz is different from specified rate 200.0000 Hz.


Exported stream 0 (TEMP1) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data/6_Data/TEMP1_C.csv
  Source ID: MD-V5-0001023
  Data shape: (23513, 1)
  Sample count: 23513

Exported stream 1 (PPG_RED) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data/6_Data/PPG_RED_C.csv
  Source ID: MD-V5-0001023
  Data shape: (78166, 1)
  Sample count: 78166

Exported stream 2 (TEMP1) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data/6_Data/TEMP1_I.csv
  Source ID: MD-V6-0000405
  Data shape: (25349, 1)
  Sample count: 25349

Exported stream 5 (PPG_IR) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data/6_Data/PPG_IR_I.csv
  Source ID: MD-V6-0000405
  Data shape: (84098, 1)
  Sample count: 84098

Exported stream 6 (PPG_IR) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data/6_Data/PPG_IR_C.csv
  Source ID: MD-V5-0001023
  Data shape: (78166, 1)
  Sample count: 78166

Exported stream 7 (PPG_RED) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMU

...

Exported stream 0 (PPG_GRN) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data/57_Data/PPG_GRN_C.csv
  Source ID: MD-V5-0001023
  Data shape: (76795, 1)
  Sample count: 76795

Exported stream 1 (PPG_GRN) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data/57_Data/PPG_GRN_I.csv
  Source ID: MD-V6-0000405
  Data shape: (76640, 1)
  Sample count: 76640

Exported stream 2 (PPG_IR) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data/57_Data/PPG_IR_C.csv
  Source ID: MD-V5-0001023
  Data shape: (76795, 1)
  Sample count: 76795

Exported stream 3 (PPG_RED) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data/57_Data/PPG_RED_C.csv
  Source ID: MD-V5-0001023
  Data shape: (76795, 1)
  Sample count: 76795

Exported stream 4 (EDA) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data/57_Data/EDA_I.csv
  Source ID: MD-V6-0000405
  Data shape: (46208, 1)
  Sample count: 46208

Exported stream 5 (TEMP1) to /Users/debbiehsu/Documents/PythonProjects/GTRI

Stream 9: Calculated effective sampling rate 142.9372 Hz is different from specified rate 200.0000 Hz.


Exported stream 0 (PPG_RED) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data/58_Data/PPG_RED_C.csv
  Source ID: MD-V5-0001023
  Data shape: (73513, 1)
  Sample count: 73513

Exported stream 1 (PPG_GRN) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data/58_Data/PPG_GRN_I.csv
  Source ID: MD-V6-0000405
  Data shape: (73363, 1)
  Sample count: 73363

Exported stream 2 (HR) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data/58_Data/HR_I.csv
  Source ID: MD-V6-0000405
  Data shape: (2171, 1)
  Sample count: 2171

Exported stream 3 (PPG_IR) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data/58_Data/PPG_IR_C.csv
  Source ID: MD-V5-0001023
  Data shape: (73513, 1)
  Sample count: 73513

Exported stream 4 (THERM) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data/58_Data/THERM_I.csv
  Source ID: MD-V6-0000405
  Data shape: (22117, 1)
  Sample count: 22117

Exported stream 5 (HR) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Dat

Stream 2: Calculated effective sampling rate 152.6520 Hz is different from specified rate 200.0000 Hz.


Exported stream 0 (PPG_RED) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data/60_Data/PPG_RED_I.csv
  Source ID: MD-V6-0000405
  Data shape: (67162, 1)
  Sample count: 67162

Exported stream 1 (THERM) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data/60_Data/THERM_C.csv
  Source ID: MD-V5-0001023
  Data shape: (20244, 1)
  Sample count: 20244

Exported stream 2 (PPG_IR) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data/60_Data/PPG_IR_C.csv
  Source ID: MD-V5-0001023
  Data shape: (67286, 1)
  Sample count: 67286

Exported stream 3 (HR) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data/60_Data/HR_I.csv
  Source ID: MD-V6-0000405
  Data shape: (1901, 1)
  Sample count: 1901

Exported stream 4 (EDA) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data/60_Data/EDA_C.csv
  Source ID: MD-V5-0001023
  Data shape: (40487, 1)
  Sample count: 40487

Exported stream 6 (THERM) to /Users/debbiehsu/Documents/PythonProjects/GTRI_MHMUAV/Data/60_

In [38]:
# --- Alternative to the full process_batch() call above: reprocess just one participant ---
# combined_df = reprocess_participant(25, xdf_folder, metadata_csv, physio_output_path)

### Step 2 — Merge Survey Data

Reads the raw physio CSV and joins per-block survey scores. Output is saved to `output_path`.

In [51]:
combined_df = pd.read_csv(physio_output_path)   # always read raw physio — no survey columns
survey_df = pd.read_csv(survey_csv)
combined_df = merge_survey_data(combined_df, survey_df)
combined_df.to_csv(output_path, index=False)



=== SURVEY MERGE SUMMARY ===
  ID 4 (Distrust Priming): 3 block(s) detected
  ID 6 (Distrust Priming): 3 block(s) detected
  ID 7 (Trust Priming): 3 block(s) detected
  ID 10 (Trust Priming): 3 block(s) detected
  ID 12 (Trust Priming): 3 block(s) detected
  ID 13 (Distrust Priming): 2 block(s) detected
  ID 14 (Trust Priming): 3 block(s) detected
  ID 15 (Distrust Priming): 3 block(s) detected
  ID 16 (Trust Priming): 3 block(s) detected
  ID 17 (Distrust Priming): 3 block(s) detected
  ID 18 (Trust Priming): 3 block(s) detected
  ID 19 (Distrust Priming): 3 block(s) detected
  ID 20 (Trust Priming): 3 block(s) detected
  ID 21 (Distrust Priming): 3 block(s) detected
  ID 22 (Trust Priming): 3 block(s) detected
  ID 23 (Distrust Priming): 3 block(s) detected
  ID 24 (Trust Priming): 3 block(s) detected
  ID 25 (Distrust Priming): 3 block(s) detected  (3,330 row(s) tagged Restart, excluded)
  ID 26 (Trust Priming): 3 block(s) detected
  ID 27 (Distrust Priming): 3 block(s) detected
  

---

## 6. Inspection & Export

### Inspect Participant

Loads a single participant and prints `DataSyncMarker` transitions to verify block labelling. Set `inspect_id` to the participant you want to check.

In [43]:
# --- Inspect a single participant ---
inspect_id = 13

df_inspect = pd.read_csv(physio_output_path)
df_inspect = df_inspect[df_inspect['ID'] == inspect_id].reset_index(drop=True)

print(f"ID={inspect_id}: {len(df_inspect)} rows")
print(f"DataSyncMarker unique values: {sorted(df_inspect['DataSyncMarker'].dropna().unique())}\n")

# Show every state transition with row index
print("DataSyncMarker transitions:")
prev = None
for i, val in df_inspect['DataSyncMarker'].items():
    val_int = int(float(val)) if pd.notna(val) else None
    if val_int != prev:
        print(f"  row {i:>6}: {prev} → {val_int}")
        prev = val_int

# Show rows around each occurrence of value 3
if 3 in df_inspect['DataSyncMarker'].values:
    rows_with_3 = df_inspect[df_inspect['DataSyncMarker'] == 3].index.tolist()
    print(f"\nValue=3 appears in {len(rows_with_3)} rows. Context around first occurrence:")
    first = rows_with_3[0]
    print(df_inspect.loc[max(0, first-3):first+3, ['DataSyncMarker', 'Reliability', 'timestamp']])

df_inspect


ID=13: 34792 rows
DataSyncMarker unique values: [np.float64(1.0), np.float64(2.0)]

DataSyncMarker transitions:
  row   3865: None → 1
  row  18979: 1 → 2
  row  27366: 2 → 1


,ID,Condition,Date,Time,timestamp,DataSyncMarker,DataSyncMarker_channel_1,Reliability,PPG_GRN_C,PPG_RED_C,...,Neon Companion_Neon Gaze_OpticalAxisX_right,Neon Companion_Neon Gaze_OpticalAxisY_right,Neon Companion_Neon Gaze_OpticalAxisZ_right,PPG_GRN_IS,PPG_RED_IS,PPG_IR_IS,EDA_IS,HR_IS,TEMP_IS,THERM_IS
0,13,Distrust Priming,2025-08-20,14:20:47.407,2924489.538,NaN,NaN,NaN,3574.0,85123.0,...,-0.126875,-0.136087,0.982503,1112.0,36615.0,127533.0,0.030147,NaN,33.880,NaN
1,13,Distrust Priming,2025-08-20,14:20:47.448,2924489.579,NaN,NaN,NaN,3605.0,85126.0,...,-0.124039,-0.133463,0.983231,1130.0,36587.0,127484.0,0.030147,NaN,33.880,30.865
2,13,Distrust Priming,2025-08-20,14:20:47.488,2924489.619,NaN,NaN,NaN,3603.0,85134.0,...,-0.126176,-0.128759,0.983579,1116.0,36584.0,127347.0,0.030147,NaN,33.880,30.865
3,13,Distrust Priming,2025-08-20,14:20:47.528,2924489.659,NaN,NaN,NaN,3592.0,85128.0,...,-0.114808,-0.126962,0.985227,1117.0,36633.0,127226.0,0.030147,NaN,33.880,30.865
4,13,Distrust Priming,2025-08-20,14:20:47.568,2924489.699,NaN,NaN,NaN,3591.0,85126.0,...,-0.127226,-0.123224,0.984158,1116.0,36647.0,127107.0,0.030147,NaN,33.948,30.975
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34787,13,Distrust Priming,2025-08-20,14:44:02.770,2925884.901,1.0,1.755715e+09,LL,3656.0,84783.0,...,0.167194,0.436725,0.883751,1062.0,41134.0,135898.0,0.030146,44.82,38.251,35.014
34788,13,Distrust Priming,2025-08-20,14:44:02.811,2925884.942,1.0,1.755715e+09,LL,3654.0,84770.0,...,0.167194,0.436725,0.883751,1047.0,41140.0,135898.0,0.030146,44.82,38.251,35.014
34789,13,Distrust Priming,2025-08-20,14:44:02.851,2925884.982,1.0,1.755715e+09,LL,3658.0,84795.0,...,0.167194,0.436725,0.883751,1061.0,41119.0,135898.0,0.030146,44.82,38.251,35.014
34790,13,Distrust Priming,2025-08-20,14:44:02.891,2925885.022,1.0,1.755715e+09,LL,3671.0,84807.0,...,0.167194,0.436725,0.883751,1064.0,41100.0,135898.0,0.030146,44.82,38.250,34.986


### Export Filtered Data

Filter `combined_df` by any column (`ID`, `Condition`, `Reliability`, etc.) and export to CSV. Uncomment the relevant example call below.

In [41]:
def export_merged_data_by_group(df, output_dir=".", **filters):
    """
    Export a filtered subset of merged_data to CSV.

    Parameters
    ----------
    df         : DataFrame  — the merged combined DataFrame (e.g. combined_df)
    output_dir : str        — directory for the exported CSV (created if needed)
    **filters  : column=value keyword filters; value may be a scalar or list
                 e.g. Condition="Trust Priming"
                      ID=[11, 23, 27]
                      Condition="Distrust Priming", ID=16
                      Reliability="HH"

    Returns
    -------
    Filtered DataFrame (also written to disk as merged_data_<filter_labels>.csv)
    """
    if not filters:
        print("No filters specified. Pass at least one keyword argument, e.g. Condition='Trust Priming'")
        return None

    mask = pd.Series(True, index=df.index)
    label_parts = []

    for col, value in filters.items():
        if col not in df.columns:
            available = [c for c in df.columns]
            print(f"WARNING: Column '{col}' not found. Available columns:\n  {available}")
            continue
        if isinstance(value, (list, tuple)):
            mask &= df[col].isin(value)
            label_parts.append(f"{col}_{'_'.join(str(v).replace(' ', '-') for v in value)}")
        else:
            mask &= df[col] == value
            label_parts.append(f"{col}_{str(value).replace(' ', '-')}")

    filtered = df[mask].reset_index(drop=True)

    if filtered.empty:
        print(f"No rows matched the specified filters: {filters}")
        return filtered

    os.makedirs(output_dir, exist_ok=True)
    filename = "merged_data_" + "__".join(label_parts) + ".csv"
    output_path = os.path.join(output_dir, filename)
    filtered.to_csv(output_path, index=False)

    print(f"Exported {len(filtered):,} rows x {len(filtered.columns)} columns")
    print(f"Filters applied : {filters}")
    if 'ID' in filtered.columns:
        ids = sorted(filtered['ID'].dropna().astype(int).unique().tolist())
        print(f"Participant IDs : {ids}")
    if 'Condition' in filtered.columns:
        print(f"Condition(s)    : {filtered['Condition'].unique().tolist()}")
    print(f"Saved to        : {output_path}")

    return filtered


In [45]:
# --- Export merged_data by group ---
# Load the combined data if not already in memory
#combined_df = pd.read_csv(output_path)

# Export a single condition
# export_merged_data_by_group(combined_df, output_dir=".", Condition="Trust Priming")
# export_merged_data_by_group(combined_df, output_dir=".", Condition="Distrust Priming")

# Export specific participant(s)
#export_merged_data_by_group(combined_df, output_dir=".", ID=13)
# export_merged_data_by_group(combined_df, output_dir=".", ID=[11, 23, 27])

# Export a condition + reliability block combo
# export_merged_data_by_group(combined_df, output_dir=".", Condition="Trust Priming", Reliability="HH")

# Export all conditions separately in a loop
# for cond in combined_df["Condition"].dropna().unique():
#     export_merged_data_by_group(combined_df, output_dir=".", Condition=cond)


Exported 34,792 rows x 69 columns
Filters applied : {'ID': 13}
Participant IDs : [13]
Condition(s)    : ['Distrust Priming']
Saved to        : ./merged_data_ID_13.csv


,ID,Condition,Date,Time,timestamp,DataSyncMarker,DataSyncMarker_channel_1,Video_Condition,Reliability,PPG_GRN_C,...,Neuroticism_IS,Openness_IS,Propensity_IS,NARS_IS,Cohesion_IS,NASA_TLX_IS,Prediction_IS,MDMT_North_IS,MDMT_South_IS,MDMT_Human_IS
0,13,Distrust Priming,2025-08-20,14:20:47.407,2924489.538,NaN,NaN,NaN,NaN,3574.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,13,Distrust Priming,2025-08-20,14:20:47.448,2924489.579,NaN,NaN,NaN,NaN,3605.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,13,Distrust Priming,2025-08-20,14:20:47.488,2924489.619,NaN,NaN,NaN,NaN,3603.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,13,Distrust Priming,2025-08-20,14:20:47.528,2924489.659,NaN,NaN,NaN,NaN,3592.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,13,Distrust Priming,2025-08-20,14:20:47.568,2924489.699,NaN,NaN,NaN,NaN,3591.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34787,13,Distrust Priming,2025-08-20,14:44:02.770,2925884.901,1.0,1.755715e+09,NaN,LL,3656.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34788,13,Distrust Priming,2025-08-20,14:44:02.811,2925884.942,1.0,1.755715e+09,NaN,LL,3654.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34789,13,Distrust Priming,2025-08-20,14:44:02.851,2925884.982,1.0,1.755715e+09,NaN,LL,3658.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34790,13,Distrust Priming,2025-08-20,14:44:02.891,2925885.022,1.0,1.755715e+09,NaN,LL,3671.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
